In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": False, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["park_and_ride", "park_and_ride_est_bundle"], "daysim-validation-notebook": ["all_tour_mode", "all_trip_mode", "auto_ownership", "day_pattern", "escort_tour_mode", "escort_trip_mode", "households", "intermediate_stop_generation", "other_home_based_tour_mode", "other_home_based_trip_mode", "persons", "school_location", "school_tour_mode", "school_trip_mode", "telecommute", "time_choice", "tours", "tour_destination", "transit_pass_ownership", "trips", "trip_destination", "workbased_subtour_generation", "workbased_subtour_mode", "work_location", "work_tour_mode", "work_trip_mode"], "network-validation-notebook": ["transit_validation", "traffic_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "//modelstation2/c$/workspace/sc_10_16_26", "survey_directories": {"survey": "R:/e2projects_two/2023_base_year/2023_survey/activitysim_format_20260629/skims_attached"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "mode_list": ["sov_inc2", "hov2_inc2", "hov3_inc2", "walk", "bike", "trnst", "ferry", "commuter_rail", "litrat"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}, "output_dir": "//modelstation2/c$/workspace/sc_10_16_26/asim_output", "data_dir": "//modelstation2/c$/workspace/sc_10_16_26/asim_data"}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "network_2023_asim", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/activitysim", "abm_model": "activitysim", "uv_directory": "C:/workspace/uv_envs/activitysim", "run_accessibility_calcs": True, "run_setup_emme_project_folders": True, "run_setup_emme_bank_folders": True, "run_copy_scenario_inputs": True, "run_import_networks": True, "run_skims_and_paths_free_flow": True, "run_skims_and_paths": True, "run_truck_model": True, "run_supplemental_trips": True, "run_abm": True, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_telecommute": True, "use_truck_tables": False, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": False, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import pandas as pd
import polars as pl
import geopandas as gpd
import math
from pathlib import Path
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

# table format
pd.options.display.float_format = '{:0,.0f}'.format
format_percent = "{:,.1%}".format

In [3]:
# read data
tour_data = util.get_tour_data(summary_config)
# trip_data = util.get_trip_data(summary_config)
land_use = util.get_landuse_data(summary_config)

In [4]:
# get all drive to transit tours
tour_data_pnr = tour_data[tour_data["tour_mode"] == "DRIVE_TRN"].\
    merge(land_use[["zone_id", "TAZ", "PNR_SPACES"]], left_on="pnr_zone_id", right_on="zone_id", how="left").\
    rename(columns={'TAZ': 'pnr_taz'})

## Park and Ride Tours

In [ ]:
# get skims for survey records

survey = pd.read_parquet("T:/60day-TEMP/Stefan/park_and_ride_lot_choice/park_and_ride_lot_choice_choosers_combined.parquet")
estimation_bundle = pd.read_parquet("T:/60day-TEMP/Stefan/park_and_ride_lot_choice/park_and_ride_lot_choice_alternatives_combined.parquet")

survey = tour_data_pnr[(tour_data_pnr['source']=='survey') & (tour_data_pnr['pnr_zone_id']>0)].\
    merge(estimation_bundle, left_on=['tour_id','pnr_zone_id'], right_on=['tour_id','pnr_zone_id'], how='inner').\
    rename(
        columns={
            # 'pnr_zone_id': 'pnr_zone_id_original',
            'pnr_olt_time': 'pnr_olt_time_survey',
            'override_choice': 'pnr_zone_id',
            'util_drive_time': 'pnr_olt_time',
            'util_total_wait': 'pnr_ldt_wait',
            'util_walk_time': 'pnr_ldt_walk',
            'util_ivtt': 'pnr_ldt_ivtt'
        })

In [6]:
# get skims for model records

model_pnr_skims = pd.read_parquet(Path(summary_config['output_dir']) / "final_tours.parquet",
                                  columns = ['tour_id', 'pnr_olt_time',
                                             'pnr_lot_time',
                                             'pnr_ldt_ivtt', 'pnr_dlt_ivtt', 'pnr_ldt_wait', 'pnr_dlt_wait',
                                             'pnr_ldt_walk', 'pnr_dlt_walk'])
# model_pnr = model[model["tour_mode"] == "DRIVE_TRN"]

model = tour_data_pnr[(tour_data_pnr['source']=='model') & (tour_data_pnr['tour_mode'] == 'DRIVE_TRN')].\
    rename(
        columns={
            'pnr_olt_time': 'pnr_olt_time_original', 
        }).\
    merge(model_pnr_skims, on='tour_id', how='left').\
    rename(
        columns={
            'util_ivtt': 'pnr_ldt_ivtt', 
            'util_total_wait': 'pnr_ldt_wait', 
            'util_walk_time': 'pnr_ldt_walk', 
            # 'pnr_lot_time', 'pnr_dlt_ivtt','pnr_dlt_wait', 'pnr_dlt_walk',
        })

In [7]:
import numpy as np
def create_time_bin_120mins(col):
    max_bin = 120
    bin_size = 10
    labels = [str(i) for i in range(0, max_bin, bin_size)]

    return pd.cut(
        col,
        bins=np.arange(0, max_bin + bin_size, bin_size),
        labels=labels,
        include_lowest=True,
        right=False,
    )

In [8]:
tour_pnr_est_bundle = pd.concat([survey,model])

tour_pnr_est_bundle['pnr_ldt_wait_bin'] = create_time_bin_120mins(tour_pnr_est_bundle['pnr_ldt_wait'])
tour_pnr_est_bundle['pnr_ldt_walk_bin'] = create_time_bin_120mins(tour_pnr_est_bundle['pnr_ldt_walk'])
tour_pnr_est_bundle['pnr_ldt_ivtt_bin'] = create_time_bin_120mins(tour_pnr_est_bundle['pnr_ldt_ivtt'])

In [9]:
util.plot_share_barchart(tour_pnr_est_bundle, 'tour_weight', 'pnr_olt_time_bin', title="drive time: home to park and ride lot")

In [10]:
util.plot_share_barchart(tour_pnr_est_bundle, 'tour_weight', 'pnr_ldt_ivtt_bin', title="in-vehicle time: park and ride lot to destination")

In [11]:
util.plot_share_barchart(tour_pnr_est_bundle, 'tour_weight', 'pnr_ldt_walk_bin', title="walk time: park and ride lot to destination")

In [12]:
util.plot_share_barchart(tour_pnr_est_bundle, 'tour_weight', 'pnr_ldt_wait_bin', title="wait time: park and ride lot to destination")